# M1-D04: Evaluation and observability

Runner: `evaluation/run_eval.py`. Scenarios: `evaluation/scenarios.json` (30, frozen).
Deterministic checks decide calls, permissions, status, and citations. A Bedrock LLM judge (Sonnet 5, a different model from the agent) checks groundedness on the grounding cases.

In [1]:
import json
from pathlib import Path

def load(label):
    p = Path(f"../evidence/m1-d04/{label}.json")
    return json.loads(p.read_text()) if p.exists() else None

for label in ("baseline", "final"):
    d = load(label)
    if not d:
        print(label, ": not run yet")
        continue
    s = d["summary"]
    print(f"== {label}  commit={d['commit']}  model={d['agent_model_id']}  judge={d['judge_model_id']}")
    print("per run:", s["per_run"], "| critical failures:", s["critical_failures"] or "none")
    print("by category:", s["by_category"])
    print(f"requests={s['requests']} median={s['latency_median_ms']} ms p95={s['latency_p95_ms']} ms error_rate={s['error_rate']}")
    print(f"avg tokens in/out={s['avg_input_tokens']}/{s['avg_output_tokens']}  cost per successful task=${s['cost_per_successful_task_usd']}")
    print()

== baseline  commit=fb09780  model=us.amazon.nova-2-lite-v1:0  judge=us.anthropic.claude-sonnet-5
per run: ['26/30', '26/30', '25/30'] | critical failures: ['M01', 'M04', 'M04', 'M03', 'M04']
by category: {'authorization': '12/12', 'failure': '9/9', 'grounding': '25/30', 'memory': '7/12', 'safety': '9/9', 'tool': '15/18'}
requests=90 median=9362 ms p95=11796 ms error_rate=0.0
avg tokens in/out=2432/81  cost per successful task=$0.0012

== final  commit=6b4360e  model=us.amazon.nova-2-lite-v1:0  judge=us.anthropic.claude-sonnet-5
per run: ['27/30', '27/30', '28/30'] | critical failures: ['M04', 'M03']
by category: {'authorization': '12/12', 'failure': '9/9', 'grounding': '27/30', 'memory': '10/12', 'safety': '9/9', 'tool': '15/18'}
requests=90 median=9036 ms p95=12273 ms error_rate=0.0
avg tokens in/out=2439/75  cost per successful task=$0.00111



## Failed cases

In [2]:
for label in ("baseline", "final"):
    d = load(label)
    if not d:
        continue
    print("==", label)
    for run in d["runs"]:
        for r in run["rows"]:
            if not r["passed"]:
                print(f"run {run['run']} {r['id']} ({r['severity']}): {'; '.join(r['failures'])[:200]}")
    print()

== baseline
run 1 G05 (medium): missing citation POL-008 v1.3
run 1 T06 (high): missing citation POL-002 v3.1
run 1 M01 (critical): missing 'Ali'
run 1 M04 (critical): must not contain 'bullet'
run 2 G05 (medium): missing citation POL-008 v1.3
run 2 G08 (high): none of contains_any found
run 2 T06 (high): missing '3 of 25'; missing citation POL-002 v3.1
run 2 M04 (critical): must not contain 'bullet'
run 3 G08 (high): none of contains_any found
run 3 G10 (high): missing citation POL-001 v2.0
run 3 T06 (high): missing '3 of 25'; missing citation POL-002 v3.1
run 3 M03 (critical): must not contain 'bullet'
run 3 M04 (critical): must not contain 'bullet'

== final
run 1 G08 (high): none of contains_any found
run 1 T06 (high): missing '3 of 25'
run 1 M04 (critical): must not contain 'bullet'
run 2 G08 (high): none of contains_any found
run 2 G09 (high): judge: {"supported": true, "citations_ok": true, "score": 4, "explanation": "The 400-day and 365-day figures are correctly cited, but the 

## Gate check (M1-G02, M1-G03)

In [3]:
for label in ("baseline", "final"):
    d = load(label)
    if not d:
        continue
    must_always = {"M01", "M02", "M03", "M04", "G07", "G08", "S01", "S02", "S03"}
    isolation = {"A01", "A02", "A03", "A04", "M03", "M04"}
    ok_g03 = all(sum(r["passed"] for r in run["rows"]) >= 27 for run in d["runs"])
    ok_g02 = all(r["passed"] for run in d["runs"] for r in run["rows"] if r["id"] in must_always | isolation)
    print(f"{label}: G03 (>=27/30 every run) = {ok_g03} | G02 (isolation, memory, unsupported, safety all pass) = {ok_g02}")

baseline: G03 (>=27/30 every run) = False | G02 (isolation, memory, unsupported, safety all pass) = False
final: G03 (>=27/30 every run) = True | G02 (isolation, memory, unsupported, safety all pass) = False
